## Install OCI CLI (if not already installed)
```bash
bash -c "$(curl -L https://raw.githubusercontent.com/oracle/oci-cli/master/scripts/install/install.sh)"
```

In [ ]:
!which oci

In [ ]:
!oci session refresh --profile harleycalvert

# Set Up OCI Terraform

[Set Up OCI Terraform](https://docs.oracle.com/en-us/iaas/developer-tutorials/tutorials/tf-provider/01-summary.htm)

## Install OpenTofu

In [ ]:
!snap install --classic opentofu

In [ ]:
!tofu -v

## Create SSH Key for Peer VMs

Both peer1 and peer2 VMs share the same key pair.  
If you already created `~/.ssh/peer_vm_key` when running peer1, skip this cell.

# Create SSH keys - Run this in Terminal
ssh-keygen -t rsa -b 4096 -f ~/.ssh/peer_vm_key

In [ ]:
!cat ~/.ssh/peer_vm_key.pub

## Structure

```
peer2/
  main.tf          variables.tf     outputs.tf
  provider.tf      terraform.tfvars
  modules/
    compartment/   main.tf  variables.tf  outputs.tf
    vcn/           main.tf  security_lists.tf  variables.tf  outputs.tf
    compute/       main.tf  variables.tf  outputs.tf
```

**VCN CIDRs (must not overlap):**
- peer1 VCN: `10.1.0.0/16`
- peer2 VCN: `10.2.0.0/16`

**LPG peering:** peer1 creates the acceptor LPG; peer2 connects to it via `peer_lpg_id`.

In [ ]:
!mkdir -p modules/compartment modules/vcn modules/compute

In [ ]:
%%writefile ./provider.tf

provider "oci" {
  config_file_profile = "harleycalvert"
}


# Main / Root

## Workflow

1. Apply **peer1** first and note its `lpg_id` output.
2. Set `peer_lpg_id` in this file's `terraform.tfvars` to that value.
3. Apply **peer2** (this notebook) — this establishes the peering connection.


In [ ]:
%%writefile ./main.tf

module "compartment" {
  source           = "./modules/compartment"
  tenancy_id       = var.tenancy_id
  compartment_name = var.compartment_name
}

module "vcn" {
  source             = "./modules/vcn"
  compartment_id     = module.compartment.id
  vcn_cidr           = var.vcn_cidr
  public_subnet_cidr = var.public_subnet_cidr
  peer_vcn_cidr      = var.peer_vcn_cidr
  admin_ip           = var.admin_ip
  peer_lpg_id        = var.peer_lpg_id
}

module "compute" {
  source              = "./modules/compute"
  tenancy_id          = var.tenancy_id
  compartment_id      = module.compartment.id
  subnet_id           = module.vcn.public_subnet_id
  assign_public_ip    = true
  ssh_public_key_path = var.ssh_public_key_path
  display_name        = var.vm_name
  hostname            = var.vm_name
  shape               = var.shape
  image_id            = var.image_id
}


In [ ]:
%%writefile ./variables.tf

variable "tenancy_id" {
  type        = string
  description = "OCID of the tenancy"
}

variable "compartment_name" {
  type        = string
  description = "Name of the OCI compartment to create"
}

variable "vcn_cidr" {
  type        = string
  description = "CIDR block for this VCN"
}

variable "public_subnet_cidr" {
  type        = string
  description = "CIDR block for the public subnet"
}

variable "peer_vcn_cidr" {
  type        = string
  description = "CIDR block of the peer VCN"
}

variable "shape" {
  type        = string
  description = "Compute instance shape"
}

variable "image_id" {
  type        = string
  description = "OCID of the compute instance image"
}

variable "admin_ip" {
  type        = string
  description = "Admin public IP in CIDR format"
}

variable "ssh_public_key_path" {
  type        = string
  description = "Path to SSH public key for VM access"
}

variable "vm_name" {
  type        = string
  description = "Display name and hostname for the VM"
}

variable "peer_lpg_id" {
  type        = string
  default     = ""
  description = "LPG OCID from the peer VCN. Leave empty for peer1; set to peer1 lpg_id output for peer2"
}


In [ ]:
%%writefile ./terraform.tfvars

# this file is not submitted to Git - specify in .gitignore

tenancy_id          = "<your-tenancy_id>"
compartment_name    = "vcn-peer2"
vcn_cidr            = "10.2.0.0/16"
public_subnet_cidr  = "10.2.0.0/24"
peer_vcn_cidr       = "10.1.0.0/16"
# https://docs.oracle.com/en-us/iaas/images/
# Use Canonical-Ubuntu-22.04 for your region
shape               = "VM.Standard.E2.1.Micro"
image_id            = "<ocid1.image.oc1.ap-melbourne-1.xxxx>"
admin_ip            = "<your-public-ip>/32"  #<YOUR_PUBLIC_IP>/32
ssh_public_key_path = "~/.ssh/peer_vm_key.pub"
vm_name             = "peer2-vm"
peer_lpg_id         = "<ocid1.localpeeringgateway.oc1.ap-melbourne-1.xxxxxx-peer1>"  # set this to peer1's lpg_id output, then apply


In [ ]:
%%writefile ./outputs.tf

output "lpg_id" {
  value       = module.vcn.lpg_id
  description = "Peer1: copy this into peer2 terraform.tfvars as peer_lpg_id"
}

output "vm_public_ip" {
  value = module.compute.public_ip
}

output "vm_private_ip" {
  value = module.compute.private_ip
}


# Compartment

In [ ]:
%%writefile ./modules/compartment/main.tf

resource "oci_identity_compartment" "compartment" {
  compartment_id = var.tenancy_id
  name           = var.compartment_name
  description    = "Managed by OpenTofu"
  enable_delete  = true
}


In [ ]:
%%writefile ./modules/compartment/variables.tf

variable "tenancy_id" {
  type = string
}

variable "compartment_name" {
  type = string
}


In [ ]:
%%writefile ./modules/compartment/outputs.tf

output "id" {
  value = oci_identity_compartment.compartment.id
}


# VCN + Local Peering Gateway (LPG)

An **LPG** connects two VCNs in the same OCI region without traffic leaving Oracle's network.

- Each VCN gets one LPG.
- The route table directs peer-VCN traffic via the LPG instead of the IGW.
- **Peer1** is the *acceptor* — its LPG just waits.
- **Peer2** is the *requestor* — it sets `peer_id` to peer1's LPG OCID, which establishes the connection.

In [ ]:
%%writefile ./modules/vcn/main.tf

resource "oci_core_vcn" "vcn" {
  compartment_id = var.compartment_id
  cidr_block     = var.vcn_cidr
  display_name   = "vcn"
  dns_label      = "vcn"
}

resource "oci_core_internet_gateway" "igw" {
  compartment_id = var.compartment_id
  vcn_id         = oci_core_vcn.vcn.id
  display_name   = "igw"
  enabled        = true
}

resource "oci_core_local_peering_gateway" "lpg" {
  compartment_id = var.compartment_id
  vcn_id         = oci_core_vcn.vcn.id
  display_name   = "lpg"
  peer_id        = var.peer_lpg_id != "" ? var.peer_lpg_id : null
}

resource "oci_core_route_table" "public_rt" {
  compartment_id = var.compartment_id
  vcn_id         = oci_core_vcn.vcn.id
  display_name   = "public-rt"

  route_rules {
    destination       = "0.0.0.0/0"
    destination_type  = "CIDR_BLOCK"
    network_entity_id = oci_core_internet_gateway.igw.id
  }

  route_rules {
    destination       = var.peer_vcn_cidr
    destination_type  = "CIDR_BLOCK"
    network_entity_id = oci_core_local_peering_gateway.lpg.id
  }
}

resource "oci_core_subnet" "public_subnet" {
  compartment_id  = var.compartment_id
  vcn_id          = oci_core_vcn.vcn.id
  cidr_block      = var.public_subnet_cidr
  display_name    = "public-subnet"
  route_table_id  = oci_core_route_table.public_rt.id
  dhcp_options_id = data.oci_core_dhcp_options.default.options[0].id
  dns_label       = "public"

  security_list_ids = [oci_core_security_list.public_sl.id]
}

data "oci_core_dhcp_options" "default" {
  compartment_id = var.compartment_id
  vcn_id         = oci_core_vcn.vcn.id
  display_name   = "Default DHCP Options for vcn"
}


In [ ]:
%%writefile ./modules/vcn/security_lists.tf

resource "oci_core_security_list" "public_sl" {
  compartment_id = var.compartment_id
  vcn_id         = oci_core_vcn.vcn.id
  display_name   = "public-subnet-sl"

  # --- INGRESS RULES ---

  # Allow SSH from admin IP
  ingress_security_rules {
    protocol = "6"
    source   = var.admin_ip

    tcp_options {
      min = 22
      max = 22
    }
  }

  # Allow all ICMP from peer VCN (enables ping between VMs)
  ingress_security_rules {
    protocol = "1"
    source   = var.peer_vcn_cidr
  }

  # Allow ICMP fragmentation-needed from anywhere (required for Path MTU Discovery)
  ingress_security_rules {
    protocol = "1"
    source   = "0.0.0.0/0"

    icmp_options {
      type = 3
      code = 4
    }
  }

  # --- EGRESS RULES ---

  egress_security_rules {
    protocol    = "all"
    destination = "0.0.0.0/0"
  }
}


In [ ]:
%%writefile ./modules/vcn/variables.tf

variable "compartment_id" {
  type = string
}

variable "vcn_cidr" {
  type = string
}

variable "public_subnet_cidr" {
  type = string
}

variable "peer_vcn_cidr" {
  type        = string
  description = "CIDR block of the peer VCN for routing and security rules"
}

variable "admin_ip" {
  type        = string
  description = "Admin public IP in CIDR format"
}

variable "peer_lpg_id" {
  type        = string
  default     = ""
  description = "LPG OCID from the peer VCN. Leave empty for peer1 (acceptor); set to peer1 lpg_id for peer2 (requestor)"
}


In [ ]:
%%writefile ./modules/vcn/outputs.tf

output "vcn_id" {
  value = oci_core_vcn.vcn.id
}

output "public_subnet_id" {
  value = oci_core_subnet.public_subnet.id
}

output "lpg_id" {
  value       = oci_core_local_peering_gateway.lpg.id
  description = "Copy this into peer2 terraform.tfvars as peer_lpg_id"
}


# Compute

Single VM per VCN. No NSG needed — security list handles SSH and ICMP.

In [ ]:
%%writefile ./modules/compute/main.tf

data "oci_identity_availability_domains" "ads" {
  compartment_id = var.tenancy_id
}

resource "oci_core_instance" "vm" {
  availability_domain = data.oci_identity_availability_domains.ads.availability_domains[var.ad_index].name
  compartment_id      = var.compartment_id
  display_name        = var.display_name
  shape               = var.shape

  source_details {
    source_type = "image"
    source_id   = var.image_id
  }

  create_vnic_details {
    subnet_id        = var.subnet_id
    assign_public_ip = var.assign_public_ip
    hostname_label   = replace(var.hostname, "-", "")
  }

  metadata = {
    ssh_authorized_keys = file(pathexpand(var.ssh_public_key_path))
  }

  preserve_boot_volume = false
}


In [ ]:
%%writefile ./modules/compute/variables.tf

variable "tenancy_id" {
  type = string
}

variable "compartment_id" {
  type = string
}

variable "subnet_id" {
  type = string
}

variable "ssh_public_key_path" {
  type = string
}

variable "display_name" {
  type = string
}

variable "hostname" {
  type = string
}

variable "shape" {
  type = string
}

variable "image_id" {
  type = string
}

variable "assign_public_ip" {
  type    = bool
  default = true
}

variable "ad_index" {
  type    = number
  default = 0
}


In [ ]:
%%writefile ./modules/compute/outputs.tf

output "instance_id" {
  value = oci_core_instance.vm.id
}

output "public_ip" {
  value = oci_core_instance.vm.public_ip
}

output "private_ip" {
  value = oci_core_instance.vm.private_ip
}


In [ ]:
!tree -al

In [ ]:
!tofu init

In [ ]:
!tofu validate

In [ ]:
!tofu plan

In [ ]:
!tofu apply -auto-approve

In [ ]:
!tofu output

## Test the peering

After both peer1 and peer2 are applied:
```bash
ssh -i ~/.ssh/peer_vm_key ubuntu@<peer2_vm_public_ip>
ping <peer1_vm_private_ip>   # e.g. 10.1.0.x
```


## Connect to the VM
```bash
ssh -i ~/.ssh/peer_vm_key ubuntu@<peer2_vm_public_ip>
```

In [ ]:
!tofu destroy -auto-approve